---
**Author:** Leonardo Gabriel Mourao Thiel  
**Project:** Master Thesis – System Inertia in the Energy System of the Future:
Model-Based Cost Optimization to Secure Inertia Requirements

**Notebook:**  Optimization Model for Energy System Analysis (2040 Scenario)


**Date:** 27.04.2026  
---

# Optimization Model for Energy System Analysis (2040 Scenario)

This notebook implements and solves a large-scale optimization model 
for the analysis of future energy system scenarios in the year 2040.

The model is formulated as a mathematical optimization problem and 
solved using the Gurobi Optimizer. It captures key aspects of power 
system operation, including generation dispatch, system inertia, 
and cross-border electricity flows.

---

## Objective

The primary objective of the model is to determine an optimal dispatch 
of generation units under different system configurations, while 
ensuring system stability and operational constraints.

Special emphasis is placed on:

- the role of system inertia (H_sys)
- the integration of renewable energy sources
- the impact of virtual inertia technologies
- cross-country interactions within the European power system

---

## Model Structure

The optimization model consists of:

- **Decision variables** representing generation, storage, and system states  
- **Constraints** ensuring physical feasibility and system stability  
- **Objective function** minimizing system cost (or another defined metric)  

The model is solved for multiple scenarios, allowing a comparative 
analysis of different system configurations.

---

## Scenarios

The following scenarios are analyzed:

- **No inertia**: baseline case without inertia constraints  
- **Thermal only**: inertia provided by conventional generation  
- **Thermal + virtual**: inclusion of virtual inertia technologies  

---

## Workflow

The notebook is structured as follows:

1. Initialization of the optimization environment  
2. Definition of model parameters and input data  
3. Formulation of decision variables and constraints  
4. Model solution using Gurobi  
5. Extraction and analysis of results  

---

## Reproducibility

The model relies on external data sources and a valid Gurobi license.
To ensure reproducibility, configuration parameters and data paths 
are defined explicitly within the notebook.

---

## Notes

This notebook focuses on the implementation of the optimization model.
Subsequent notebooks are used for post-processing, visualization,
and interpretation of the results.

# Optimization Model Setup

This section initializes the optimization environment using Gurobi.
The required packages are imported, and the solver is configured
with the appropriate license parameters.

The model will be used to solve the energy system optimization problem.

In [1]:
"""
Initialization of the optimization environment.

This section installs and imports the Gurobi optimizer,
and configures the license parameters required to run the model.
"""

import sys
# Install requirements
!{sys.executable} -m pip install requirements.txt

# Import Gurobi library
import gurobipy as gp
from gurobipy import Model, GRB, quicksum
from inputs import InputConfig, InputLoader
import os
import gurobipy as gp
from build_model import build_full_model
import pandas as pd

# ---------------------------------------------------------
# Gurobi license configuration
# ---------------------------------------------------------

# Parameters required for accessing the Gurobi Web License Service (WLS)
# NOTE: In a production or published environment, credentials should be
# stored securely (e.g., environment variables) and not hardcoded.
def create_gurobi_env():
    access_id = os.getenv("GUROBI_ACCESS_ID")

    if access_id:
        params = {
            "WLSACCESSID": access_id,
            "WLSSECRET": os.getenv("GUROBI_SECRET"),
            "LICENSEID": int(os.getenv("GUROBI_LICENSE_ID")),
        }
        return gp.Env(params=params)
    else:
        # fallback: lokale Lizenz
        return gp.Env()

env = create_gurobi_env()


ERROR: Could not find a version that satisfies the requirement requirements.txt (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\Leo\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for requirements.txt


HINT: You are attempting to install a package literally named "requirements.txt" (which cannot exist). Consider using the '-r' flag to install the packages listed in requirements.txt
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2710734
Academic license 2710734 - for non-commercial use only - registered to le___@student.kit.edu


## 1. Input Parameters and Cost Assumptions

This section defines the core input parameters of the optimization model,
including the temporal scope, geographical coverage, and cost assumptions
for different inertia-providing technologies.

The cost parameters are annualized and scaled to the selected simulation
period to ensure consistency with the modeled time horizon.

The following technologies are considered:

- battery-based inertia  
- solar + battery hybrid systems  
- wind-based inertia  
- solar (assumed to provide no inertia contribution)

These cost assumptions directly influence the optimization outcome,
as they determine the relative competitiveness of inertia-providing options.

In [2]:

# ---------------------------------------------------------
# General model parameters
# ---------------------------------------------------------

# Base directory for input data
path = "../Data"

# List of countries included in the model (European system)
countryList = [
    "AL","AT","BA","BE","BG","CH","CZ","DE","DK","ES","FR","GR"
]
countryList += [
    "HR","HU","IT","LU","MK","ME","NL","PL","PT","RO","RS","SI","SK"
]

# Define simulation time horizon (summer period 2040)
start_date = "2040-07-01"
end_date   = "2040-07-02"

# Compute duration of the simulation period in days
duration = (pd.to_datetime(end_date) - pd.to_datetime(start_date)).days


# ---------------------------------------------------------
# Inertia cost parameters (scaled to simulation horizon)
# ---------------------------------------------------------

# Annualized cost assumptions (€/MW or similar unit)
# are scaled proportionally to the modeled time horizon

# Cost of providing inertia via battery systems
inertia_costs_battery = 2230 * (duration / 365)

# Cost of hybrid solar + battery systems providing inertia
inertia_costs_solar_battery = 3220 * (duration / 365)

# Cost of wind-based inertia provision
inertia_costs_wind = 1670 * (duration / 365)

# Solar-only systems assumed to provide no inertia (cost = 0)
inertia_costs_solar = 0

## 2. Data Input and Preprocessing

In this section, all required input data for the optimization model is loaded 
using a configuration-based approach.

The `InputConfig` object defines the temporal scope, data location, and 
geographical coverage of the model. Based on this configuration, the 
`InputLoader` retrieves and preprocesses all relevant datasets.

The resulting `inputs` object serves as the central data container for the model 
and includes, among others:

- demand time series (`df_load`)
- generation unit data (`thermal_units`, etc.)
- system parameters and constraints

This structured approach ensures consistency and modularity in data handling.

In [3]:

# ---------------------------------------------------------
# Create input configuration
# ---------------------------------------------------------

# The configuration object defines:
# - data path (location of input files)
# - time horizon of the simulation
# - set of countries included in the model
cfg = InputConfig(
    path,
    start_date,
    end_date,
    countryList
)

# ---------------------------------------------------------
# Load and preprocess input data
# ---------------------------------------------------------

# The InputLoader reads raw data files and transforms them into
# structured formats required by the optimization model.
# This typically includes:
# - filtering by date range
# - selecting relevant countries
# - organizing data into accessible attributes

inputs = InputLoader.load(cfg)

# The resulting object 'inputs' acts as a centralized data container
# for the optimization model, e.g.:
# inputs.df_load         -> demand time series
# inputs.thermal_units   -> thermal generation unit parameters
# inputs.renewables      -> renewable generation data (if available)
# inputs.network         -> transmission or flow constraints (if included)

## 3. Scenario-Based Optimization Framework

This section defines and executes the optimization model for multiple
energy system scenarios.

Each scenario represents a different configuration of inertia provision:

- **thermal_plus_virtual**: inertia from both conventional generation and virtual sources  
- **thermal_only**: inertia provided exclusively by thermal units  
- **no_inertia**: no inertia constraints considered  

The model is built and solved separately for each scenario using a shared
data structure and parameter set. This enables a consistent comparison
of system behavior under different assumptions.

Solver parameters are explicitly configured to control numerical stability,
solution quality, and computational performance.

In [4]:



# =========================
# Scenario definitions
# =========================

# Each scenario specifies whether inertia constraints and/or
# virtual inertia contributions are included in the model
SCENARIOS = [
    {
        "name": "thermal_plus_virtual",
        "calculate_inertia": True,
        "calculate_virtual_inertia": True,
    },
    {
        "name": "no_inertia",
        "calculate_inertia": False,
        "calculate_virtual_inertia": False,
    },
    {
        "name": "thermal_only",
        "calculate_inertia": True,
        "calculate_virtual_inertia": False,
    },
]


# =========================
# Solver parameter configuration
# =========================

def set_parameters(model, nodefile_dir, log_path):
    """
    Configure Gurobi solver parameters.

    Parameters
    ----------
    model : gurobipy.Model
        Optimization model instance
    nodefile_dir : str
        Directory for node file storage (not explicitly used here)
    log_path : str
        Path to the solver log file

    Returns
    -------
    model : gurobipy.Model
        Model with configured parameters
    """

    # Limit number of threads 
    model.setParam("Threads", 8)

    # Acceptable optimality gap (1%)
    model.setParam("MIPGap", 0.01)
    model.setParam("MIPFocus", 1)
    # Disable presolve reductions for full model transparency
    model.setParam("Presolve", 2)
    model.setParam("PrePasses", 40)
    model.setParam("Heuristics", 0.2)
    model.setParam("Aggregate", 2)

    # Improve numerical robustness
    model.setParam("NumericFocus", 1)

    # Solver method configuration
    model.setParam("Method", 0)      # primal simplex
    model.setParam("NodeMethod", 1)  # dual simplex for nodes

    # Log file for each scenario
    model.setParam("LogFile", log_path)

    return model


# =========================
# Main execution function
# =========================

def run_scenarios(
    env,
    inputs,
    countries,
    nodefile_dir,
    inertia_costs_battery,
    inertia_costs_solar_battery,
    inertia_costs_solar,
    inertia_costs_wind,
    out_dir="..\\Results\\gurobi_out",
    write_lp=False,
):
    """
    Build and solve the optimization model for all defined scenarios.

    Parameters
    ----------
    env : gurobipy.Env
        Gurobi environment (license + configuration)
    inputs : object
        Structured input data container
    countries : list
        List of countries included in the model
    nodefile_dir : str
        Directory for node file storage (optional)
    inertia_costs_* : float
        Cost parameters for different inertia technologies
    out_dir : str
        Output directory for results and logs
    write_lp : bool
        Whether to export LP files

    Returns
    -------
    dict
        Dictionary containing solution statistics per scenario
    """

    # Ensure output directory exists
    os.makedirs(out_dir, exist_ok=True)

    results = {}

    # =========================
    # Loop over all scenarios
    # =========================

    for sc in SCENARIOS:
        name = sc["name"]
        print(f"\n=== Running scenario: {name} ===")

        # Initialize model with shared environment
        model = gp.Model(env=env)

        # Apply solver configuration
        model = set_parameters(
            model,
            nodefile_dir,
            os.path.join(out_dir, f"{name}.log"),
        )

        # Build optimization model (variables, constraints, objective)
        model = build_full_model(
            model,
            inputs,
            countries,
            inertia_costs_battery=inertia_costs_battery,
            inertia_costs_solar_battery=inertia_costs_solar_battery,
            inertia_costs_solar=inertia_costs_solar,
            inertia_costs_wind=inertia_costs_wind,
            calculate_inertia=sc["calculate_inertia"],
            calculate_virtual_inertia=sc["calculate_virtual_inertia"],
        )

        # Export LP formulation (for debugging / transparency)
        if write_lp:
            model.write(os.path.join(out_dir, f"{name}.lp"))

        # =========================
        # Solve optimization problem
        # =========================
        model.optimize()

     
        # =========================
        # Save outputs
        # =========================

        if model.SolCount > 0:
            # Save solution file
            model.write(os.path.join(out_dir, f"{name}.sol"))

        elif model.status in (GRB.INFEASIBLE, GRB.INF_OR_UNBD):
            # Compute and export IIS for infeasibility analysis
            model.computeIIS()
            model.write(os.path.join(out_dir, f"{name}.ilp"))

        # Free memory
        model.dispose()

    return results

## 4. Model Execution

In this section, the optimization model is executed for all defined scenarios.

The previously defined input data, cost parameters, and configuration settings
are passed to the scenario runner. Each scenario is solved independently,
and key solution metrics are collected.

The results include:

- solver status  
- number of feasible solutions  
- objective value  
- model size (variables and constraints)  

These outputs provide a first validation of model feasibility and performance.

In [5]:
# ---------------------------------------------------------
# Execute optimization for all scenarios
# ---------------------------------------------------------

# Note: 'env' should be initialized beforehand using Gurobi license parameters
# e.g., env = gp.Env(params=params)

results = run_scenarios(
    env=env,                          # Gurobi environment (license + settings)
    inputs=inputs,                    # preprocessed input data
    countries=countryList,            # set of modeled countries
    nodefile_dir=path,                # directory for solver node files (if used)

    # Cost parameters for inertia-providing technologies
    inertia_costs_battery=inertia_costs_battery,
    inertia_costs_solar_battery=inertia_costs_solar_battery,
    inertia_costs_solar=inertia_costs_solar,
    inertia_costs_wind=inertia_costs_wind
)




=== Running scenario: thermal_plus_virtual ===
Set parameter Threads to value 8
Set parameter MIPGap to value 0.01
Set parameter MIPFocus to value 1
Set parameter Presolve to value 2
Set parameter PrePasses to value 40
Set parameter Heuristics to value 0.2
Set parameter Aggregate to value 2
Set parameter NumericFocus to value 1
Set parameter Method to value 0
Set parameter NodeMethod to value 1
Set parameter LogFile to value "..\Results\gurobi_out\thermal_plus_virtual.log"
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-7267U CPU @ 3.10GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 8 threads

         Reduce the value of the Threads parameter to improve performance


Non-default parameters:
MIPGap  0.01
Method  0
Heuristics  0.2
MIPFocus  1
NodeMethod  1
Aggregate  2
NumericFocus  1
PrePasses  40
Presolve  2
Threads  8

Academic license 2710734 - for non-commercia

Variable types: 25218 continuous, 24060 integer (24047 binary)
Performing another presolve...
Presolve removed 1 rows and 927 columns
Presolve time: 0.32s
Root relaxation presolve removed 576 rows and 590 columns
Root relaxation presolved: 23903 rows, 47761 columns, 95652 nonzeros


Use crossover to convert LP symmetric solution to basic solution...
Crossover time: 0.03 seconds (0.01 work units)

Root relaxation: objective 1.417964e+07, 7586 iterations, 0.24 seconds (0.13 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 1.4180e+07    0    4          - 1.4180e+07      -     -    1s
H    0     0                    1.419189e+07 1.4180e+07  0.09%     -    1s

Explored 1 nodes (7767 simplex iterations) in 1.58 seconds (1.07 work units)
Thread count was 8 (of 4 available processors)

Solution count 1: 1.41919e+07 

Optimal solution found (tolerance 1.00e-02)
Best ob